# F2-M00: Índice Fase 2 — EDA Datos Originales

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Genera la **página índice de la Fase 2** con KPIs globales del dataset
y tarjetas de navegación a los 10 módulos de análisis exploratorio.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/fase2_index.html`

## Flujo
```
M00 Índice → M01 Inspección → M02 Calidad → M03 Nulos →
M04 Univariante Num → M05 Univariante Cat → M06 Evolución →
M07 Conclusiones
```

## Siguiente
`f2_m01_inspeccion.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    # ROOT robusto: busca src/ subiendo niveles desde el directorio actual
    _cwd = Path.cwd()
    ROOT = _cwd
    for _ in range(10):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError(
            f"No se encontró carpeta 'src/' desde {_cwd}. "
            f"Verifica que el notebook está dentro de AU_UJI/"
        )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
import pandas as pd

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])  # Crea si no existen

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y CALCULAR KPIs
# ============================================================================
# Lee df_alumno.parquet y extrae métricas globales para la cabecera
# del índice: registros, alumnos únicos, variables, período.
# ============================================================================

print('=' * 60)
print('ÍNDICE FASE 2: EDA DATOS ORIGINALES')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# --- Métricas globales ---
n_filas = len(df)
n_cols = len(df.columns)
n_alumnos = df['per_id_ficticio'].nunique() if 'per_id_ficticio' in df.columns else n_filas

# --- Detectar rango de cursos académicos ---
curso_col = None
for col in ['curso_aca', 'curso_aca_id']:
    if col in df.columns:
        curso_col = col
        break

if curso_col:
    curso_min = df[curso_col].min()
    curso_max = df[curso_col].max()
    rango_cursos = f"{curso_min}-{curso_max}"
else:
    rango_cursos = "N/D"

# --- Resumen ---
print(f'✅ Dataset: {formato_numero_es(n_filas)} filas × {n_cols} columnas')
print(f'👥 Alumnos únicos: {formato_numero_es(n_alumnos)}')
print(f'📅 Período: {rango_cursos}')

ÍNDICE FASE 2: EDA DATOS ORIGINALES
✅ Dataset: 109.568 filas × 37 columnas
👥 Alumnos únicos: 30.872
📅 Período: 2010-2020


In [3]:
# ============================================================================
# CELDA 3: GENERAR HTML — PÁGINA ÍNDICE FASE 2
# ============================================================================
# Genera fase2_index.html con:
# - KPIs globales del dataset
# - Descripción del objetivo de la fase
# - Tarjetas de navegación a los 8 módulos (M00-M07)
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Navegación dinámica ---
nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa="fase2",
    modulo_activo="indice"
)

# --- KPIs ---
KPIS = [
    {'valor': formato_numero_es(n_filas), 'titulo': 'Registros'},
    {'valor': formato_numero_es(n_alumnos), 'titulo': 'Alumnos'},
    {'valor': str(n_cols), 'titulo': 'Variables'},
    {'valor': rango_cursos, 'titulo': 'Período'},
]
kpis_html = generar_kpis_html(KPIS)

# --- Sección: Objetivo de la Fase ---
seccion_intro = generar_seccion_html(
    titulo="Objetivo de la Fase",
    contenido='''
        <p>Esta fase realiza el <strong>Análisis Exploratorio de Datos (EDA)</strong> sobre los datos
        originales, antes de cualquier transformación o agregación.</p>
        <p>El objetivo es comprender la estructura, calidad y distribución de los datos para
        identificar patrones, anomalías y guiar las decisiones de Feature Engineering en la Fase 3.</p>
        <div style="background:#EBF8FF; padding:15px; border-radius:8px; margin-top:15px; border-left:4px solid #3182ce;">
            <strong>📌 Nota:</strong> Los datos mantienen su estructura longitudinal
            (múltiples filas por alumno). La agregación a 1 fila por alumno se realizará en Fase 3.
        </div>
    ''',
    icono="🎯"
)

# --- Sección: Tarjetas de módulos ---
# Cada tarjeta enlaza al HTML del módulo correspondiente.
# 8 módulos completados.

modulos_html = '''
<div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(280px, 1fr)); gap:20px;">

    <!-- M01: Inspección -->
    <a href="m01_inspeccion.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #3182ce; transition:all 0.2s;">
            <h3 style="color:#3182ce; margin:0 0 8px 0;">🔍 M01: Inspección</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Estructura, tipos de datos, memoria, valores únicos</p>
        </div>
    </a>

    <!-- M02: Calidad -->
    <a href="m02_calidad.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #38a169; transition:all 0.2s;">
            <h3 style="color:#38a169; margin:0 0 8px 0;">✅ M02: Calidad</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Duplicados, anomalías de texto, distribución general</p>
        </div>
    </a>

    <!-- M03: Nulos -->
    <a href="m03_nulos.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #ed8936; transition:all 0.2s;">
            <h3 style="color:#ed8936; margin:0 0 8px 0;">❓ M03: Nulos</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Análisis de valores faltantes, patrones y correlaciones</p>
        </div>
    </a>

    <!-- M04: Univariante Numérico -->
    <a href="m04_univariante_num.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #805ad5; transition:all 0.2s;">
            <h3 style="color:#805ad5; margin:0 0 8px 0;">📈 M04: Univariante Numérico</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Distribuciones, estadísticas, outliers de variables numéricas</p>
        </div>
    </a>

    <!-- M05: Univariante Categórico -->
    <a href="m05_univariante_cat.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #e53e3e; transition:all 0.2s;">
            <h3 style="color:#e53e3e; margin:0 0 8px 0;">📊 M05: Univariante Categórico</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Frecuencias, cardinalidad, dominancia de variables categóricas</p>
        </div>
    </a>

    <!-- M06: Evolución -->
    <a href="m06_evolucion.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #319795; transition:all 0.2s;">
            <h3 style="color:#319795; margin:0 0 8px 0;">📈 M06: Evolución</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Evolución temporal, tendencias por curso académico</p>
        </div>
    </a>

    <!-- M07: Conclusiones -->
    <a href="m07_conclusiones.html" style="text-decoration:none;">
        <div style="background:#f8fafc; border-radius:10px; padding:20px; border-left:4px solid #2c5282; transition:all 0.2s;">
            <h3 style="color:#2c5282; margin:0 0 8px 0;">📝 M07: Conclusiones</h3>
            <p style="color:#718096; margin:0; font-size:0.9em;">Resumen de hallazgos, próximos pasos, reglas de negocio</p>
        </div>
    </a>

    </a>

    </a>

    </a>

</div>
'''

seccion_modulos = generar_seccion_html(
    titulo="Módulos de la Fase",
    contenido=modulos_html,
    icono="📋"
)

# --- Ensamblar y guardar HTML ---
contenido_html = kpis_html + seccion_intro + seccion_modulos

html_completo = render_base_html(
    titulo="Fase 2: EDA Datos Originales",
    icono="📊",
    subtitulo="Análisis Exploratorio de Datos Originales",
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f2_m00_indice.ipynb',
    notebook_carpeta='fase2_eda'
)

ruta_html = RUTA_FASE2 / "fase2_index.html"
guardar_html(html_completo, ruta_html)
print(f"\n✅ HTML: {ruta_html}")

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\fase2_index.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\fase2_index.html


In [4]:
# ============================================================================
# CELDA 4: RESUMEN DE EJECUCIÓN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M00 ÍNDICE COMPLETADO')
print('=' * 60)
print(f'📁 HTML: fase2_index.html')
print(f'📊 Módulos activos: M01-M07')
print(f'✅ Fase 2: 8 módulos completados (M00-M07)')
print(f'📌 Siguiente: f2_m01_inspeccion.ipynb')


✅ F2-M00 ÍNDICE COMPLETADO
📁 HTML: fase2_index.html
📊 Módulos activos: M01-M07
✅ Fase 2: 8 módulos completados (M00-M07)
📌 Siguiente: f2_m01_inspeccion.ipynb
